# Data Preprocessing Pipeline

End-to-end cleaning, harmonisation, merging, and imputation of multi-source macroeconomic panel data (OECD, Eurostat, World Bank, Gallup).

In [43]:
# ── Installation (run once) ─────────────────────────────────────────────
import subprocess, sys

for pkg in ["pandas", "pycountry", "openpyxl", "numpy"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

In [44]:
# ── Core imports ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import pycountry
import warnings
import os

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

## 1 · Country-code helpers

Two utility functions to normalise country identifiers to ISO 3166-1 alpha-3.

In [45]:
def iso2_to_iso3(code: str):
    """Convert ISO-2 alpha code (e.g. 'FR') to ISO-3 alpha code (e.g. 'FRA')."""
    try:
        return pycountry.countries.get(alpha_2=str(code).strip()).alpha_3
    except Exception:
        return None


def name_to_iso3(name: str):
    """Convert a country name string to ISO-3 alpha code."""
    try:
        return pycountry.countries.lookup(str(name).strip()).alpha_3
    except Exception:
        return None

## 2 · Load and clean structured CSV sources

All files share the schema `[country, year, value]`. A single `clean_dataset()` function handles: column standardisation, deduplication, type coercion, ISO-2 to ISO-3 conversion (Eurostat only), and per-country time-interpolation + forward/back-fill.

In [46]:
# ── File registry: filename -> needs_iso2_fix ────────────────────────────
DATA_DIR = "data/raw/"

CSV_FILES = {
    "Eurostat_Public_Debt_GDP_2000_2025.csv": True,
    "Global_Public_Debt_GDP_2000_2025.csv": False,
    "Global_Energy_Dependency_0_100.csv": False,
    "OECD_GDP_Growth_2000_2025.csv": False,
    "OECD_Inflation_CPI_2000_2025.csv": False,
    "OECD_Productivity_Variation_2000_2025.csv": False,
    "OECD_RD_GDP_2000_2025.csv": False,
    "OECD_Tertiary_Education_2000_2025.csv": False,
    "OECD_Unemployment_Rate_2000_2025.csv": False,  # ISO-2 codes
}



def clean_dataset(df: pd.DataFrame, fix_iso2: bool = False) -> pd.DataFrame:
    """
    Standardise a raw [country, year, value] CSV.

    Steps:
      1. Lowercase and strip column names.
      2. Keep only [country, year, value].
      3. Coerce year to int, value to float.
      4. Drop exact duplicates.
      5. Optionally convert ISO-2 country codes to ISO-3.
      6. Drop rows with unresolvable country.
      7. Sort by [country, year].
      8. Per-country linear interpolation, then forward/back-fill.
    """
    df = df.copy()
    df.columns = df.columns.str.lower().str.strip()

    missing_cols = {"country", "year", "value"} - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    df = df[["country", "year", "value"]].copy()
    df["year"]  = pd.to_numeric(df["year"],  errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["year"])
    df["year"] = df["year"].astype(int)
    df = df.drop_duplicates()
    df["country"] = df["country"].str.strip()

    if fix_iso2:
        df["country"] = df["country"].apply(iso2_to_iso3)

    df = df.dropna(subset=["country"])
    df = df.sort_values(["country", "year"]).reset_index(drop=True)

    df["value"] = (
        df.groupby("country")["value"]
          .transform(
              lambda x: x
              .interpolate(method="linear", limit_direction="both")
              .ffill()
              .bfill()
          )
    )

    return df


# ── Load and clean all CSV sources ───────────────────────────────────────
datasets: dict = {}

for fname, needs_iso2_fix in CSV_FILES.items():
    raw = pd.read_csv(os.path.join(DATA_DIR, fname))
    key = fname.replace(".csv", "")
    datasets[key] = clean_dataset(raw, fix_iso2=needs_iso2_fix)

print(f"Loaded {len(datasets)} datasets:")
for k, v in datasets.items():
    print(f"  {k}: {v.shape}  |  countries: {v['country'].nunique()}")

Loaded 9 datasets:
  Eurostat_Public_Debt_GDP_2000_2025: (676, 3)  |  countries: 26
  Global_Public_Debt_GDP_2000_2025: (1833, 3)  |  countries: 114
  Global_Energy_Dependency_0_100: (3151, 3)  |  countries: 145
  OECD_GDP_Growth_2000_2025: (1314, 3)  |  countries: 52
  OECD_Inflation_CPI_2000_2025: (1362, 3)  |  countries: 54
  OECD_Productivity_Variation_2000_2025: (4132, 3)  |  countries: 44
  OECD_RD_GDP_2000_2025: (1127, 3)  |  countries: 49
  OECD_Tertiary_Education_2000_2025: (1003, 3)  |  countries: 51
  OECD_Unemployment_Rate_2000_2025: (1392, 3)  |  countries: 59


## 3 · Merge CSV sources into a wide panel

Each dataset is pivoted to (year x country), converted back to long format with an `indicator` tag, then assembled into a single (year, country) x indicator matrix.

In [47]:
# ── Pivot each dataset wide (year x country) ─────────────────────────────
pivot_datasets: dict = {}

for name, df in datasets.items():
    pivot_datasets[name] = (
        df.pivot_table(index="year", columns="country", values="value", aggfunc="mean")
          .sort_index()
    )

# ── Melt to long, tag by indicator, concatenate ───────────────────────────
dfs_long = []
for name, df in pivot_datasets.items():
    temp = df.reset_index().melt(id_vars="year", var_name="country", value_name="value")
    temp["indicator"] = name
    dfs_long.append(temp)

df_all = pd.concat(dfs_long, ignore_index=True)

# ── Final wide panel: (year, country) x indicator ─────────────────────────
final_df = (
    df_all
    .pivot_table(index=["year", "country"], columns="indicator", values="value")
    .reset_index()
)

print(f"Panel shape: {final_df.shape}")
print(f"Years: {final_df['year'].min()} - {final_df['year'].max()}")
print(f"Countries: {final_df['country'].nunique()}")
final_df.head()

Panel shape: (4675, 11)
Years: 2000 - 2025
Countries: 210


indicator,year,country,Eurostat_Public_Debt_GDP_2000_2025,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Unemployment_Rate_2000_2025
0,2000,AGO,NaN,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000,ALB,NaN,46.5182,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000,ARE,NaN,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000,ARG,NaN,0.0000,NaN,-0.7890,NaN,NaN,0.3925,NaN,NaN
4,2000,ARM,NaN,71.1911,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4 · Panel-level imputation

**Strategy (two passes, per country):**
1. Linear time-interpolation — respects the temporal structure of the panel.
2. Country-mean fill — handles leading/trailing NaNs not covered by interpolation.

In [48]:
final_clean = final_df.copy()
numeric_cols = final_clean.select_dtypes(include="number").columns.difference(["year"])

# Pass 1 — linear interpolation within each country-time series
final_clean[numeric_cols] = final_clean.groupby("country")[numeric_cols].transform(
    lambda x: x.interpolate(method="linear", limit_direction="both")
)

# Pass 2 — country-mean fill for residual edge NaNs
final_clean[numeric_cols] = final_clean.groupby("country")[numeric_cols].transform(
    lambda x: x.fillna(x.mean())
)

missing_share = final_clean[numeric_cols].isnull().mean().round(4)
print("Remaining missing share per indicator after imputation:")
print(missing_share.to_string())
final_clean.head()

Remaining missing share per indicator after imputation:
indicator
Eurostat_Public_Debt_GDP_2000_2025      0.8554
Global_Energy_Dependency_0_100          0.2847
Global_Public_Debt_GDP_2000_2025        0.4667
OECD_GDP_Growth_2000_2025               0.7110
OECD_Inflation_CPI_2000_2025            0.6999
OECD_Productivity_Variation_2000_2025   0.7553
OECD_RD_GDP_2000_2025                   0.7281
OECD_Tertiary_Education_2000_2025       0.7202
OECD_Unemployment_Rate_2000_2025        0.6742


indicator,year,country,Eurostat_Public_Debt_GDP_2000_2025,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Unemployment_Rate_2000_2025
0,2000,AGO,NaN,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000,ALB,NaN,46.5182,69.1922,NaN,NaN,NaN,NaN,NaN,NaN
2,2000,ARE,NaN,0.0000,1.8033,NaN,NaN,NaN,NaN,NaN,NaN
3,2000,ARG,NaN,0.0000,NaN,-0.7890,34.2772,NaN,0.3925,16.7119,7.5610
4,2000,ARM,NaN,71.1911,50.0284,NaN,NaN,NaN,NaN,NaN,NaN


## 5 · Load Worldwide Governance Indicators (Excel)

Keep only year >= 2000, resolve country names to ISO-3, and retain only the three columns needed: `country`, `year`, `Governance score (0-100)`.

In [49]:
dfx = pd.read_excel(os.path.join(DATA_DIR, "Worldwide_Governance_Indicators.xlsx"))
dfx = dfx.rename(columns={"Governance score (0-100)": "Worldwide_Governance_Indicators"})

# Filter to study period
dfx = dfx[dfx["Year"] >= 2000].copy()

# Resolve country names to ISO-3
dfx["Economy (name)"] = dfx["Economy (name)"].apply(name_to_iso3)
dfx = dfx.dropna(subset=["Economy (name)"])

# Rename key columns
dfx = dfx.rename(columns={"Economy (name)": "country", "Year": "year"})

# Keep only the columns required for the merge
keep_cols = ["country", "year", "Worldwide_Governance_Indicators"]
dfx = dfx[keep_cols].drop_duplicates().reset_index(drop=True)

print(f"Governance data shape: {dfx.shape}")
dfx.head()

Governance data shape: (4447, 3)


,country,year,Worldwide_Governance_Indicators
0,AND,2000,83.8269
1,AFG,2000,16.6312
2,AGO,2000,27.4713
3,ALB,2000,47.6615
4,ARE,2000,42.0816


## 6 · Merge Governance Indicators onto the panel

Inner join on `[country, year]` to retain only country-year pairs present in both the OECD/Eurostat panel and the World Bank governance data.

In [50]:
final_merged = pd.merge(
    final_clean,
    dfx,
    on=["country", "year"],
    how="inner"
)

print(f"Merged panel shape: {final_merged.shape}")
final_merged.head()

Merged panel shape: (3091, 12)


,year,country,Eurostat_Public_Debt_GDP_2000_2025,Global_Energy_Dependency_0_100,Global_Public_Debt_GDP_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Unemployment_Rate_2000_2025,Worldwide_Governance_Indicators
0,2000,AGO,NaN,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.4713
1,2000,ALB,NaN,46.5182,69.1922,NaN,NaN,NaN,NaN,NaN,NaN,47.6615
2,2000,ARE,NaN,0.0000,1.8033,NaN,NaN,NaN,NaN,NaN,NaN,42.0816
3,2000,ARG,NaN,0.0000,NaN,-0.7890,34.2772,NaN,0.3925,16.7119,7.5610,64.5902
4,2000,ARM,NaN,71.1911,50.0284,NaN,NaN,NaN,NaN,NaN,NaN,46.7813


## 7 · Rename indicator columns

All renames applied in a single mapping to prevent cascading alias bugs.

In [51]:
RENAME_MAP = {
    "Eurostat_Public_Debt_GDP_2000_2025":  "Eurostat_Public_Debt_pct_GDP",
    "Global_Public_Debt_GDP_2000_2025":    "Public_Debt_pct_GDP",
    "Global_Energy_Dependency_0_100":      "Energy_Import_Dependency",
    "OECD_GDP_Growth_2000_2025":           "GDP_Growth",
    "OECD_Inflation_CPI_2000_2025":        "Inflation_Rate",
    "OECD_Productivity_Variation_2000_2025": "Productivity_Variation",
    "OECD_RD_GDP_2000_2025": "R&D_Expenditure_pct_GDP",
    "OECD_Tertiary_Education_2000_2025":   "Tertiary_Education_Attainment",
    "OECD_Unemployment_Rate_2000_2025":    "Unemployment_Rate",
    "Worldwide_Governance_Indicators": "Gov_Score",

}

final_merged = final_merged.rename(columns=RENAME_MAP)

# Sanity check: no stale column names remain
stale = [c for c in final_merged.columns if c in RENAME_MAP]
assert not stale, f"Unresolved column names after rename: {stale}"

print("Columns after renaming:")
print(final_merged.columns.tolist())
final_merged.head()

Columns after renaming:
['year', 'country', 'Eurostat_Public_Debt_pct_GDP', 'Energy_Import_Dependency', 'Public_Debt_pct_GDP', 'GDP_Growth', 'Inflation_Rate', 'Productivity_Variation', 'R&D_Expenditure_pct_GDP', 'Tertiary_Education_Attainment', 'Unemployment_Rate', 'Gov_Score']


,year,country,Eurostat_Public_Debt_pct_GDP,Energy_Import_Dependency,Public_Debt_pct_GDP,GDP_Growth,Inflation_Rate,Productivity_Variation,R&D_Expenditure_pct_GDP,Tertiary_Education_Attainment,Unemployment_Rate,Gov_Score
0,2000,AGO,NaN,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.4713
1,2000,ALB,NaN,46.5182,69.1922,NaN,NaN,NaN,NaN,NaN,NaN,47.6615
2,2000,ARE,NaN,0.0000,1.8033,NaN,NaN,NaN,NaN,NaN,NaN,42.0816
3,2000,ARG,NaN,0.0000,NaN,-0.7890,34.2772,NaN,0.3925,16.7119,7.5610,64.5902
4,2000,ARM,NaN,71.1911,50.0284,NaN,NaN,NaN,NaN,NaN,NaN,46.7813


## 8 · Build final analysis dataset

Select the 10 indicator columns plus `year` and `country`, then remove rows where more than half the numeric columns are missing (threshold: > 5 out of 10 indicators absent).

In [52]:
INDICATOR_COLS = [
    "Eurostat_Public_Debt_pct_GDP",
    "Public_Debt_pct_GDP",
    "Energy_Import_Dependency",
    "GDP_Growth",
    "Inflation_Rate",
    "Productivity_Variation",
    "R&D_Expenditure_pct_GDP",
    "Tertiary_Education_Attainment",
    "Unemployment_Rate",
    "Gov_Score",
]

final_df3 = final_merged[["year", "country"] + INDICATOR_COLS].copy()

# Drop rows where more than half the indicators are missing
MAX_MISSING = len(INDICATOR_COLS) // 2  # 5 out of 10
final_df3 = final_df3[
    final_df3[INDICATOR_COLS].isnull().sum(axis=1) <= MAX_MISSING
].copy()

print(f"final_df3 shape: {final_df3.shape}")
print(f"Countries: {final_df3['country'].nunique()}")
print(f"Years: {final_df3['year'].min()} - {final_df3['year'].max()}")
final_df3.head()

final_df3 shape: (1152, 12)
Countries: 48
Years: 2000 - 2024


,year,country,Eurostat_Public_Debt_pct_GDP,Public_Debt_pct_GDP,Energy_Import_Dependency,GDP_Growth,Inflation_Rate,Productivity_Variation,R&D_Expenditure_pct_GDP,Tertiary_Education_Attainment,Unemployment_Rate,Gov_Score
3,2000,ARG,NaN,NaN,0.0000,-0.7890,34.2772,NaN,0.3925,16.7119,7.5610,64.5902
5,2000,AUS,NaN,29.4708,0.0000,3.3597,4.4574,52.4641,1.4698,27.4757,6.2880,82.0517
6,2000,AUT,66.6000,NaN,66.9003,3.1895,2.3449,47.0911,1.8967,24.7834,3.5020,81.1755
8,2000,BEL,109.7000,NaN,87.2200,3.7167,2.5445,53.9798,1.9362,27.0850,7.0090,82.1186
11,2000,BGR,70.7000,NaN,46.8650,NaN,10.3163,11.5493,0.4952,27.4981,16.9190,62.6429


## 9 · Missing-value diagnostics

Summarise remaining missingness before final imputation, and export country-level reports for the two sparsest indicators.

In [53]:
OUTPUT_DIR = "data/processed/"

# Overall missing rate per indicator
missing_summary = final_df3[INDICATOR_COLS].isnull().mean().rename("missing_rate").to_frame()
missing_summary["missing_count"] = final_df3[INDICATOR_COLS].isnull().sum()
print(missing_summary.sort_values("missing_rate", ascending=False).to_string())

# Aggregate missing count per country
missing_by_country = (
    final_df3.groupby("country")[INDICATOR_COLS]
             .apply(lambda x: x.isnull().sum().sum())
             .rename("total_missing")
             .reset_index()
             .sort_values("total_missing", ascending=False)
)
missing_by_country.to_csv(os.path.join(OUTPUT_DIR, "missing_by_country.csv"), index=False)
print(f"\nSaved missing_by_country.csv ({missing_by_country.shape[0]} countries)")

# Per-indicator country-level reports for the two sparsest columns
for col, label in [("Energy_Import_Dependency", "ENERGY"),
                   ("Public_Debt_pct_GDP",      "PUBLICDEBT")]:
    subset = (
        final_df3.loc[final_df3[col].isnull(), ["country", col]]
                 .drop_duplicates()
                 .reset_index(drop=True)
    )
    fname = f"missing_countries_{label}.csv"
    subset.to_csv(os.path.join(OUTPUT_DIR, fname), index=False)
    print(f"Saved {fname} ({len(subset)} countries)")

                               missing_rate  missing_count
Public_Debt_pct_GDP                  0.5833            672
Eurostat_Public_Debt_pct_GDP         0.5000            576
Productivity_Variation               0.1667            192
GDP_Growth                           0.0833             96
R&D_Expenditure_pct_GDP              0.0833             96
Inflation_Rate                       0.0417             48
Tertiary_Education_Attainment        0.0208             24
Energy_Import_Dependency             0.0000              0
Unemployment_Rate                    0.0000              0
Gov_Score                            0.0000              0

Saved missing_by_country.csv (48 countries)
Saved missing_countries_ENERGY.csv (0 countries)
Saved missing_countries_PUBLICDEBT.csv (28 countries)


## 10 · Shock-recovery feature engineering

For each structural shock episode (2007 GFC, 2019 pre-COVID baseline), compute the number of years each country took to return to its shock-year GDP growth level.

In [54]:
def calculate_recovery_time(
    df: pd.DataFrame,
    shock_year: int,
    gdp_col: str
) -> pd.DataFrame:
    """
    For each country, find the first post-shock year where gdp_col
    returns to at least its shock-year level.  Adds a column
    'recovery_{shock_year}' (NaN if never recovered within the sample).
    """
    records = []

    for country, cdf in df.groupby("country"):
        cdf = cdf.sort_values("year").copy()

        baseline_rows = cdf.loc[cdf["year"] == shock_year, gdp_col]
        if baseline_rows.empty:
            cdf[f"recovery_{shock_year}"] = np.nan
            records.append(cdf)
            continue

        baseline = baseline_rows.values[0]
        recovery_lag = np.nan

        for _, row in cdf[cdf["year"] > shock_year].iterrows():
            if row[gdp_col] >= baseline:
                recovery_lag = row["year"] - shock_year
                break

        cdf[f"recovery_{shock_year}"] = recovery_lag
        records.append(cdf)

    return pd.concat(records, ignore_index=True)


final_df3 = calculate_recovery_time(final_df3, shock_year=2007, gdp_col="GDP_Growth")
final_df3 = calculate_recovery_time(final_df3, shock_year=2019, gdp_col="GDP_Growth")

preview = (
    final_df3[["country", "year", "recovery_2007", "recovery_2019"]]
    .drop_duplicates("country")
    .head(15)
)
print("Recovery lags (years) per country:")
print(preview.to_string(index=False))

Recovery lags (years) per country:
country  year  recovery_2007  recovery_2019
    ARG  2000         3.0000         2.0000
    AUS  2000        14.0000         2.0000
    AUT  2000        14.0000         2.0000
    BEL  2000        14.0000         2.0000
    BGR  2000            NaN            NaN
    BRA  2000         3.0000         2.0000
    CAN  2000         3.0000         2.0000
    CHE  2000        14.0000         2.0000
    CHL  2000         3.0000         2.0000
    CHN  2000            NaN         2.0000
    COL  2000         4.0000         2.0000
    CRI  2000        14.0000         2.0000
    CZE  2000            NaN         2.0000
    DEU  2000         3.0000         2.0000
    DNK  2000         3.0000         2.0000


## 11 · Final imputation pass

Cascading strategy:
1. Drop countries with > 30% overall missingness (insufficient temporal coverage).
2. Country-mean imputation (preserves within-country structural level).
3. Global-mean fallback for any residual gaps.

In [55]:
numeric_cols_final = final_df3.select_dtypes(include="number").columns.difference(["year"])

# Step 1: filter countries by missingness threshold
missing_ratio = (
    final_df3.groupby("country")[numeric_cols_final]
             .apply(lambda x: x.isnull().mean().mean())
)
good_countries = missing_ratio[missing_ratio < 0.30].index
df_clean = final_df3[final_df3["country"].isin(good_countries)].copy()

print(f"Countries retained: {df_clean['country'].nunique()} / {final_df3['country'].nunique()}")

# Step 2: country-mean imputation
df_clean[numeric_cols_final] = df_clean.groupby("country")[numeric_cols_final].transform(
    lambda x: x.fillna(x.mean())
)

# Step 3: global-mean fallback
global_means = df_clean[numeric_cols_final].mean().round(4)
df_clean[numeric_cols_final] = df_clean[numeric_cols_final].fillna(global_means)

residual = df_clean[numeric_cols_final].isnull().sum().sum()
print(f"Residual missing values after final imputation: {residual}")
df_clean.head()

Countries retained: 42 / 48
Residual missing values after final imputation: 0


,year,country,Eurostat_Public_Debt_pct_GDP,Public_Debt_pct_GDP,Energy_Import_Dependency,GDP_Growth,Inflation_Rate,Productivity_Variation,R&D_Expenditure_pct_GDP,Tertiary_Education_Attainment,Unemployment_Rate,Gov_Score,recovery_2007,recovery_2019
0,2000,ARG,58.8931,57.2361,0.0000,-0.7890,34.2772,797.3907,0.3925,16.7119,7.5610,64.5902,3.0000,2.0000
1,2002,ARG,58.8931,57.2361,0.0000,-10.8945,34.2772,797.3907,0.3478,16.7119,7.5610,62.6777,3.0000,2.0000
2,2003,ARG,58.8931,57.2361,0.0000,8.8370,34.2772,797.3907,0.3668,16.7119,7.5610,64.1974,3.0000,2.0000
3,2004,ARG,58.8931,57.2361,0.0000,9.0296,34.2772,797.3907,0.4038,16.7119,7.5610,65.2543,3.0000,2.0000
4,2005,ARG,58.8931,57.2361,0.0000,8.8517,34.2772,797.3907,0.4207,17.4711,7.5610,65.7908,3.0000,2.0000


## 12 · Export final datasets

Three outputs:
- **`dataset_structural_indicators.csv`** — year, country, R&D / trust / debt / energy / education.
- **`dataset_macro_performance.csv`** — year, country, GDP growth / inflation / unemployment / productivity / governance + recovery lags.
- **`dataset_complete.csv`** — full `df_clean` with all columns.

In [56]:
STRUCTURAL_COLS = [
    "year", "country",
    "RD_Expenditure_pct_GDP",
    "Trust_Institutions",
    "Public_Debt_pct_GDP",
    "Energy_Import_Dependency",
    "Tertiary_Education_Attainment",
]

MACRO_COLS = [
    "year", "country",
    "GDP_Growth",
    "Inflation_Rate",
    "Unemployment_Rate",
    "Productivity_Variation",
    "Gov_Score",
    "recovery_2007",
    "recovery_2019",
]

df_structural = df_clean[[c for c in STRUCTURAL_COLS if c in df_clean.columns]].copy()
df_macro      = df_clean[[c for c in MACRO_COLS      if c in df_clean.columns]].copy()

df_structural.to_csv(os.path.join(OUTPUT_DIR, "dataset_structural_indicators.csv"), index=False)
df_macro.to_csv(os.path.join(OUTPUT_DIR, "dataset_macro_performance.csv"),          index=False)
df_clean.to_csv(os.path.join(OUTPUT_DIR, "dataset_complete.csv"),                   index=False)

print(f"Exported dataset_structural_indicators.csv  — {df_structural.shape}")
print(f"Exported dataset_macro_performance.csv      — {df_macro.shape}")
print(f"Exported dataset_complete.csv               — {df_clean.shape}")

Exported dataset_structural_indicators.csv  — (1008, 5)
Exported dataset_macro_performance.csv      — (1008, 9)
Exported dataset_complete.csv               — (1008, 14)
